# 01 -- Quickstart Notebook

**Goal**: load `TwoDimFMAdapter`, run a baseline single-pass, run a framework multi-round ablation, and visualize the samples.

**Time budget**: 3-5 minutes on a CPU laptop.

**Companion docs**:

- [`TUTORIAL.md`](../docs/TUTORIAL.md) -- fast on-ramp (5-10 min reading).
- [`PLUG_IN_YOUR_MODEL.md`](../docs/PLUG_IN_YOUR_MODEL.md) -- bring your own SOTA checkpoint.
- [`ALGORITHMS.md`](../docs/ALGORITHMS.md) -- full scheduler / driver / merge / blender catalog.
- [`ADAPTER_INTERFACE_SPEC.md`](../docs/ADAPTER_INTERFACE_SPEC.md) -- the eight-method Protocol contract.

### Why a notebook and not a script?

Because the framework is **CPU-runnable end-to-end on a laptop** (no GPU, no torch), the only thing that distinguishes a notebook from a script is interleaved prose. This notebook interleaves the prose with the code so you can read it top-to-bottom and copy-paste any cell into a `.py` file when you want to integrate it into your own pipeline.

### A note on this notebook's execution status

The framework's venv (`./.venv`) is **stdlib + numpy + scipy** by design -- no jupyter, no matplotlib. To execute this notebook with `nbconvert`:

```bash
PYTHONPATH=. ./.venv/Scripts/python.exe -m pip install jupyter nbconvert ipykernel
PYTHONPATH=. ./.venv/Scripts/python.exe -m jupyter nbconvert --to notebook --execute examples/01_quickstart.ipynb --output 01_quickstart.executed.ipynb
```

If you cannot install those extras, you can run each cell as a standalone Python script by copy-pasting it into a `.py` file -- the code does not depend on any notebook-only state.

### Setup

Two assumptions:

1. Your working directory is the repo root (`flowa-multistep-reinference/`).
2. `PYTHONPATH=.` is set so `import adaptive_reflow` resolves to the package.

If you opened this notebook from `examples/`, the setup cell below walks up one directory and chdir's into the repo root so the adapter's default `data/twodim_fm_*.npz` weights path resolves correctly.

In [ ]:
import hashlib
import os
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().resolve()
if (REPO_ROOT / "adaptive_reflow").exists() is False:
    REPO_ROOT = REPO_ROOT.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
os.environ.setdefault("PYTHONPATH", str(REPO_ROOT))
os.chdir(REPO_ROOT)

import numpy as np
import scipy.stats as stats

from adaptive_reflow.adapters.twodim_fm import TwoDimFMAdapter
from adaptive_reflow.algorithm.scheduler import (
    CosineAnnealScheduler,
    CodimensionSheetScheduler,
    EvidenceDrivenScheduler,
    FreeTrajScheduler,
)
from adaptive_reflow.contracts import (
    ArtifactHash, CosineScheduleConfig, FactorValue,
)

print("REPO_ROOT:", REPO_ROOT)
print("TwoDimFMAdapter:", TwoDimFMAdapter)
print("numpy:", np.__version__)

## Step 1 -- Load the canonical adapter

`TwoDimFMAdapter` is the framework's reference real-model adapter. It ships pre-trained weights under `data/` for two target distributions:

- `two_moons` -- two interlocking half-circles
- `eight_gaussians` -- eight modes on a circle of radius 2.0

Both targets share the same `3 -> 64 -> 64 -> 2` velocity-field MLP; only the target distribution differs.

In [ ]:
adapter = TwoDimFMAdapter(target="two_moons")
caps = adapter.capabilities()
print("channels:        ", caps.supported_channels)
print("channel_domains: ", {str(k): v for k, v in caps.channel_domains.items()})
print("has_restart:     ", caps.has_restart_boundary)
print("has_condition:   ", caps.has_condition_injection)

## Step 2 -- Run a baseline single-pass

The baseline is the same model, same weights, single-pass inference -- no FlowA re-inference loop. We use `TwoDimFMAdapter.generate_trajectory()` which integrates the ODE once per sample. The output is `ArrayF64` of shape `(n_trajectories, endpoints_per_trajectory, n_gen, 2)`. With `n_trajectories=1, endpoints_per_trajectory=1, n_gen=N`, we get a `(1, 1, N, 2)` block of fresh baseline endpoints -- i.e. the canonical single-pass baseline.

In [ ]:
N_BASELINE = 500
SEED = 42

traj = adapter.generate_trajectory(
    n_trajectories=1,
    endpoints_per_trajectory=1,
    n_gen=N_BASELINE,
    seed=SEED,
)
baseline_endpoints = traj.reshape(-1, 2)
print("baseline endpoints:", baseline_endpoints.shape)
print("finite:", bool(np.isfinite(baseline_endpoints).all()))
print("first sample:", baseline_endpoints[0])
print("std:", baseline_endpoints.std(axis=0))

## Step 3 -- Run a framework multi-round ablation

The framework's batched driver (`BatchedTrajectoryRunner`) takes a config (which embeds the scheduler) and runs `cycle_length` rounds of re-inference, emitting the per-round endpoints in `result.per_round_endpoints[round][trajectory]`. We use the canonical four schedulers in turn.

For this quickstart we use **3 rounds** to keep wall-clock low; the full SOTA ablation uses 20 rounds (see `tools/run_sota_2d_experiment.py`).

In [ ]:
from adaptive_reflow.algorithm.batched_runner import (
    BatchedRunnerConfig, BatchedTrajectoryRunner,
)

CYCLE_LENGTH = 3
TRAJECTORIES_PER_ROUND = 25
ENDPOINTS_PER_TRAJECTORY = 4
SEED = 42

def _build_cosine_config(rounds):
    config_hash = ArtifactHash(
        hashlib.sha256(
            repr(("cosine_no_restart", int(rounds), 0.0, 1.0)).encode("utf-8")
        ).hexdigest()
    )
    return CosineScheduleConfig(
        schedule_family="cosine_no_restart",
        cycle_length=int(rounds),
        n_min=FactorValue(0.0),
        n_max=FactorValue(1.0),
        per_channel_caps={},
        fresh_noise_floor_by_channel={},
        symmetric_delta_caps_by_channel={},
        restart_triggers_allowed=(),
        config_hash=config_hash,
        frozen_before_evaluation=True,
    )

schedulers = {
    "cosine":   CosineAnnealScheduler(_build_cosine_config(CYCLE_LENGTH)),
    "codim":    CodimensionSheetScheduler(cycle_length=CYCLE_LENGTH),
    "evidence": EvidenceDrivenScheduler(_build_cosine_config(CYCLE_LENGTH)),
    "freetraj": FreeTrajScheduler(_build_cosine_config(CYCLE_LENGTH)),
}

def _cfg_for(scheduler):
    return BatchedRunnerConfig(
        cycle_length=CYCLE_LENGTH,
        trajectories_per_round=TRAJECTORIES_PER_ROUND,
        endpoints_per_trajectory=ENDPOINTS_PER_TRAJECTORY,
        scheduler=scheduler,
        seed=SEED,
    )

results = {}
for name, sched in schedulers.items():
    runner = BatchedTrajectoryRunner(_cfg_for(sched), adapter=adapter)
    res = runner.run()
    results[name] = res
    # res.per_round_endpoints[round][trajectory] -> (endpoints_per_trajectory, 2)
    last_round = res.per_round_endpoints[-1]
    last_round_endpoints = np.concatenate(last_round, axis=0)
    print(f"{name:>10s}  n_rounds={len(res.per_round_endpoints)}  per_traj_shape={last_round[0].shape}  last_round_n={len(last_round_endpoints)}  last_round_mean={last_round_endpoints.mean(axis=0)}  W2_last_round={res.per_round_w2[-1]:.4f}")

## Step 4 -- Visualize the samples

Because matplotlib is not in the framework venv by default, the cell below uses an **ASCII scatter** fallback. If you have `matplotlib` installed, the next cell produces a publication-quality plot.

The visualization compares three populations of `x in R^2` points:

1. **baseline** (top) -- single-pass endpoints (Step 2).
2. **framework / cosine** (middle) -- last-round endpoints from Step 3.
3. **analytic target** (bottom) -- fresh samples from the `two_moons` analytic sampler.

In [ ]:
from adaptive_reflow.adapters.twodim_fm_train import sample_two_moons

def ascii_scatter(xs, ys, *, width=60, height=20, title=""):
    if len(xs) == 0:
        return f"{title}\n(empty)"
    x_lo, x_hi = float(min(xs)), float(max(xs))
    y_lo, y_hi = float(min(ys)), float(max(ys))
    if x_hi == x_lo: x_hi = x_lo + 1e-9
    if y_hi == y_lo: y_hi = y_lo + 1e-9
    grid = [[' '] * width for _ in range(height)]
    for x, y in zip(xs, ys, strict=False):
        col = int((x - x_lo) / (x_hi - x_lo) * (width - 1))
        row = int((y - y_lo) / (y_hi - y_lo) * (height - 1))
        row = height - 1 - row
        grid[row][col] = '*'
    body = '\n'.join(''.join(row) for row in grid)
    return f"{title}\n{body}"

baseline_xy = baseline_endpoints[:200]
framework_last_round = np.concatenate(results["cosine"].per_round_endpoints[-1], axis=0)
framework_xy = framework_last_round[:200]
rng = np.random.default_rng(SEED)
target_xy = sample_two_moons(200, rng)

print(ascii_scatter(baseline_xy[:, 0], baseline_xy[:, 1], title=f"baseline (single-pass), N={len(baseline_xy)}"))
print()
print(ascii_scatter(framework_xy[:, 0], framework_xy[:, 1], title=f"framework / cosine, last round, N={len(framework_xy)}"))
print()
print(ascii_scatter(target_xy[:, 0], target_xy[:, 1], title=f"analytic target (two_moons), N={len(target_xy)}"))

In [ ]:
try:
    import matplotlib.pyplot as plt
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    for ax, xy, title in zip(
        axes,
        [baseline_xy, framework_xy, target_xy],
        ["baseline (single-pass)", "framework / cosine (last round)", "analytic target"],
    ):
        ax.scatter(xy[:, 0], xy[:, 1], s=4, alpha=0.6)
        ax.set_title(title); ax.set_aspect("equal"); ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig("/tmp/twodim_quickstart.png", dpi=120)
    print("Saved /tmp/twodim_quickstart.png")
except ModuleNotFoundError:
    print("matplotlib not installed; using ASCII scatter only.")

## What success looks like

For the `two_moons` target, you should see **two interlocking arcs** in all three plots. The framework version (middle) should match the arcs more cleanly than the baseline (top) -- that's the visual signature of `selection_ratio -> 1` (paper Theorem 1).

For quantitative confirmation, run the full SOTA ablation:

```bash
PYTHONPATH=. ./.venv/Scripts/python.exe tools/run_sota_2d_experiment.py --target two_moons --quick
```

...which writes a comparison table under `docs/r4-survey/` with `selection_ratio`, `wasserstein_2d`, `support_coverage`, and `energy_distance` per scheduler x seed.

## Where to go next

| You want to... | Read |
| --- | --- |
| Plug your own SOTA flow-matching model into the engine | [`PLUG_IN_YOUR_MODEL.md`](../docs/PLUG_IN_YOUR_MODEL.md) |
| Choose a different scheduler, driver, merge operator, or blender | [`ALGORITHMS.md`](../docs/ALGORITHMS.md) |
| Understand the package layout, dependency DAG, governance invariants | [`ARCHITECTURE.md`](../ARCHITECTURE.md) |
| Implement the 8-method Protocol surface from scratch | [`ADAPTER_INTERFACE_SPEC.md`](../docs/ADAPTER_INTERFACE_SPEC.md) |
| Run the four-scheduler ablation against your checkpoint | [`docs/r4-survey/07-sota-experiment-protocol.md`](../docs/r4-survey/07-sota-experiment-protocol.md) |